# Robust Validation Lab

Цель ноутбука: отделить public-оптимизацию от private-safe устойчивости.

Проверяем текущие ключевые кандидаты через несколько независимых слоев:

- public ladder и candidate registry;
- temporal backtests для deterministic baseline;
- OOF diagnostics для GRU и second residual;
- segment robustness;
- paired bootstrap / group bootstrap;
- shift/adversarial validation между historical proxy и March-2025 target rows;
- итоговый scorecard для выбора leaderboard и private-safe решений.


In [ ]:
from __future__ import annotations

import calendar
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TRAIN_PATH = ROOT / 'data' / 'train_2.csv'
if not TRAIN_PATH.exists():
    TRAIN_PATH = ROOT / 'train_2.csv'

RESULTS_DIR = ROOT / 'results'
SUBMISSIONS_DIR = ROOT / 'submissions'
OUT_DIR = RESULTS_DIR / 'robust_validation_lab'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'РТО'
SEED = 2026
rng = np.random.default_rng(SEED)

print('ROOT:', ROOT)
print('TRAIN_PATH:', TRAIN_PATH)
print('OUT_DIR:', OUT_DIR)


## 1. Candidate Registry

Фиксируем кандидатов, которые реально важны для выбора финального решения и дальнейшего улучшения public score.


In [ ]:
CANDIDATE_IDS = [118, 150, 159, 198, 203, 204, 210, 215, 220, 221]

public_log_path = RESULTS_DIR / 'public_experiment_log.csv'
public_log = pd.read_csv(public_log_path) if public_log_path.exists() else pd.DataFrame()
if not public_log.empty:
    public_log['candidate_id_num'] = pd.to_numeric(public_log['candidate_id'], errors='coerce').astype('Int64')
    public_log['public_mape_num'] = pd.to_numeric(public_log['public_mape'], errors='coerce')
    public_focus = (
        public_log[public_log['candidate_id_num'].isin(CANDIDATE_IDS)]
        .sort_values('public_mape_num', na_position='last')
        .reset_index(drop=True)
    )
else:
    public_focus = pd.DataFrame()

candidate_notes = pd.DataFrame([
    {'candidate_id': 118, 'role': 'old transfer residual', 'risk': 'medium', 'private_read': 'independent signal; useful only if stable by segments'},
    {'candidate_id': 150, 'role': 'old fresh GRU on cal050', 'risk': 'medium', 'private_read': 'previous strong full component'},
    {'candidate_id': 159, 'role': 'old full best blend', 'risk': 'medium', 'private_read': 'fallback full solution before temporal204'},
    {'candidate_id': 198, 'role': 'sign-asym storeseason baseline', 'risk': 'low', 'private_read': 'strong pre-temporal deterministic anchor'},
    {'candidate_id': 203, 'role': 'temporal/Q1 baseline, lc004', 'risk': 'low', 'private_read': 'first temporal/Q1 public probe'},
    {'candidate_id': 204, 'role': 'temporal/Q1 baseline, lc006', 'risk': 'low', 'private_read': 'current pure baseline anchor'},
    {'candidate_id': 210, 'role': 'fresh GRU on 204', 'risk': 'medium-low', 'private_read': 'large OOF/public gain; central model layer'},
    {'candidate_id': 215, 'role': '75% 210 + 25% 118', 'risk': 'medium', 'private_read': 'public-positive transfer blend; needs segment stress'},
    {'candidate_id': 220, 'role': 'tiny second residual on 215', 'risk': 'medium', 'private_read': 'current best; tiny correction but small gain'},
    {'candidate_id': 221, 'role': 'stronger second residual on 215', 'risk': 'medium-high', 'private_read': 'same public MAPE as 220 with larger correction'},
])

if not public_focus.empty:
    registry = candidate_notes.merge(
        public_focus[['candidate_id_num', 'family', 'file', 'public_score', 'public_mape', 'notes']],
        left_on='candidate_id',
        right_on='candidate_id_num',
        how='left',
    ).drop(columns=['candidate_id_num'])
else:
    registry = candidate_notes.copy()

registry.to_csv(OUT_DIR / 'candidate_registry.csv', index=False)
display(registry.sort_values(['public_mape', 'candidate_id'], na_position='last'))


## 2. Deterministic Baseline Helpers

Реконструируем `198` и `204`, чтобы честно проверять baseline-family на historical periods.


In [ ]:
STATIC_CAT_COLS = ['Регион', 'Торговая площадь, категориальный', 'Дата открытия, категориальный']
CURRENT_LAMBDA = 0.55
VALID_PERIODS = [(2024, 3), (2024, 11), (2024, 12), (2025, 1), (2025, 2)]

def mape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

def wape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100)

def log_mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(np.log(np.clip(y_true, 1.0, None) / np.clip(y_pred, 1.0, None)))) * 100)

def safe_div(a, b):
    return pd.Series(a).astype(float) / pd.Series(b).replace(0, np.nan).astype(float)

def label(year: int, month: int) -> str:
    return f'{year:04d}-{month:02d}'

def shift(year: int, month: int, months_back: int) -> tuple[int, int]:
    stamp = pd.Timestamp(year=year, month=month, day=1) - pd.DateOffset(months=months_back)
    return int(stamp.year), int(stamp.month)

def load_wide():
    raw = pd.read_csv(TRAIN_PATH, low_memory=False)
    raw['period_label'] = pd.to_datetime(
        raw['Год'].astype(str) + '-' + raw['Месяц'].astype(str).str.zfill(2) + '-01'
    ).dt.strftime('%Y-%m')
    rto = raw.pivot_table(index='new_id', columns='period_label', values=TARGET, aggfunc='mean')
    static_cols = [c for c in STATIC_CAT_COLS if c in raw.columns]
    static = raw.sort_values(['new_id', 'Год', 'Месяц']).groupby('new_id')[static_cols].first()
    return raw, rto, static

raw, rto, static = load_wide()
print(raw.shape, rto.shape, static.shape)


In [ ]:
def target_frame(rto: pd.DataFrame, static: pd.DataFrame, year: int, month: int, include_target: bool = True) -> pd.DataFrame:
    lag1_y, lag1_m = shift(year, month, 1)
    lag2_y, lag2_m = shift(year, month, 2)
    lag3_y, lag3_m = shift(year, month, 3)
    lag12_y, lag12_m = shift(year, month, 12)
    lag13_y, lag13_m = shift(year, month, 13)
    frame = pd.DataFrame(index=rto.index)
    frame.index.name = None
    frame['new_id'] = frame.index.astype(int)
    frame['lag1'] = rto[label(lag1_y, lag1_m)]
    frame['lag2'] = rto[label(lag2_y, lag2_m)]
    frame['lag3'] = rto[label(lag3_y, lag3_m)]
    frame['lag12'] = rto[label(lag12_y, lag12_m)]
    frame['lag13'] = rto[label(lag13_y, lag13_m)]
    if include_target and label(year, month) in rto.columns:
        frame[TARGET] = rto[label(year, month)]
    frame['rolling3_mean'] = frame[['lag1', 'lag2', 'lag3']].mean(axis=1)
    frame['rolling2_pre_mean'] = frame[['lag2', 'lag3']].mean(axis=1)
    frame['momentum_raw'] = frame['lag1'] / frame['rolling3_mean'].replace(0, np.nan)
    frame['momentum_clip'] = frame['momentum_raw'].fillna(1.0).clip(0.95, 1.05)
    frame['feb_to_jan'] = frame['lag1'] / frame['lag2'].replace(0, np.nan)
    frame['jan_to_dec'] = frame['lag2'] / frame['lag3'].replace(0, np.nan)
    frame['lag1_to_lag12'] = frame['lag1'] / frame['lag12'].replace(0, np.nan)
    frame['lag1_to_lag13'] = frame['lag1'] / frame['lag13'].replace(0, np.nan)
    frame['store_mar_feb_ratio_prev'] = frame['lag12'] / frame['lag13'].replace(0, np.nan)
    frame['volatility_rel3'] = frame[['lag1', 'lag2', 'lag3']].std(axis=1) / frame['rolling3_mean'].replace(0, np.nan)
    frame['structural_decline_score'] = (
        frame['feb_to_jan'].fillna(1.0).lt(0.97)
        & frame['jan_to_dec'].fillna(1.0).lt(0.99)
        & frame['lag1_to_lag12'].fillna(1.0).lt(1.02)
    ).astype(float)
    frame = frame.join(static, on='new_id', how='left')
    return frame.reset_index(drop=True)

def historical_factor(rto: pd.DataFrame, target_year: int, target_month: int, calendar_lambda: float) -> float:
    ratios = []
    for source_year in sorted({int(c[:4]) for c in rto.columns if c.endswith(f'-{target_month:02d}')}):
        if source_year >= target_year:
            continue
        prev_y, prev_m = shift(source_year, target_month, 1)
        source = label(source_year, target_month)
        prev = label(prev_y, prev_m)
        if source not in rto.columns or prev not in rto.columns:
            continue
        ratio = rto[source] / rto[prev].replace(0, np.nan)
        if target_month == 3:
            source_feb_days = calendar.monthrange(source_year, 2)[1]
            target_feb_days = calendar.monthrange(target_year, 2)[1]
            ratio *= (1.0 - calendar_lambda) + calendar_lambda * (source_feb_days / target_feb_days)
        ratios.append(ratio)
    if not ratios:
        raise ValueError(f'No historical ratios for {target_year}-{target_month:02d}')
    return float(pd.concat(ratios).replace([np.inf, -np.inf], np.nan).dropna().median())

def sign_asym_store_pred(frame, base, factor, target_non_leap, calendar_lambda=0.55, w_pos=0.2, w_neg=1.4, clip_pos=0.02, clip_neg=0.03, vol_max=0.12):
    ratio = frame['lag12'] / frame['lag13'].replace(0, np.nan)
    if target_non_leap:
        ratio *= (1.0 - calendar_lambda) + calendar_lambda * (29.0 / 28.0)
    else:
        ratio *= (1.0 - calendar_lambda) + calendar_lambda * (28.0 / 29.0)
    ratio = ratio.replace([np.inf, -np.inf], np.nan).fillna(factor).clip(0.85, 1.45)
    delta = np.log(ratio.to_numpy(float) / factor)
    pos = np.clip(delta, 0.0, clip_pos)
    neg = np.clip(delta, -clip_neg, 0.0)
    volume = (frame['lag13'].to_numpy(float) / (frame['lag13'].to_numpy(float) + 100_000_000.0)).clip(0.0, 1.0)
    stable = (1.0 - frame['volatility_rel3'].fillna(0.0).to_numpy(float) / 0.12).clip(0.0, 1.0)
    rel = volume * stable
    if vol_max is not None:
        rel *= (frame['volatility_rel3'].fillna(0.0).to_numpy(float) <= vol_max).astype(float)
    rel *= (frame['structural_decline_score'].to_numpy(float) <= 0).astype(float)
    return np.asarray(base, dtype=float) * np.exp(rel * (w_pos * pos + w_neg * neg))

def two_month_temporal_anchor(rto, frame, year, month, clip_lo=0.85, clip_hi=1.35):
    prev1_y, prev1_m = shift(year, month, 1)
    prev2_y, prev2_m = shift(year, month, 2)
    py, pm = shift(year, month, 12)
    pprev1_y, pprev1_m = shift(year, month, 13)
    pprev2_y, pprev2_m = shift(year, month, 14)
    current_two = rto[label(prev1_y, prev1_m)] + rto[label(prev2_y, prev2_m)]
    previous_two = rto[label(pprev1_y, pprev1_m)] + rto[label(pprev2_y, pprev2_m)]
    growth = (current_two / previous_two.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    growth = growth.fillna(growth.median()).clip(clip_lo, clip_hi)
    anchor = rto[label(py, pm)] * growth
    return pd.Series(anchor, index=rto.index).reindex(frame['new_id']).to_numpy(float)

def blend_anchor(base, anchor, alpha=0.12, log_clip=0.06):
    ratio = np.log(np.maximum(anchor, 1.0) / np.maximum(base, 1.0))
    return np.asarray(base, dtype=float) * np.exp(alpha * np.clip(ratio, -log_clip, log_clip))

def predict_198_204(year, month, include_target=True):
    frame = target_frame(rto, static, year, month, include_target=include_target)
    factor = historical_factor(rto, year, month, CURRENT_LAMBDA)
    target_non_leap = month == 3 and calendar.monthrange(year, 2)[1] == 28
    base = frame['lag1'].to_numpy(float) * factor
    frame['pred198'] = sign_asym_store_pred(frame, base, factor, target_non_leap=target_non_leap)
    anchor = two_month_temporal_anchor(rto, frame, year, month, clip_lo=0.85, clip_hi=1.35)
    frame['temporal_anchor'] = anchor
    frame['pred204'] = blend_anchor(frame['pred198'].to_numpy(float), anchor, alpha=0.12, log_clip=0.06)
    frame['temporal_corr_log'] = np.log(np.clip(frame['pred204'], 1.0, None) / np.clip(frame['pred198'], 1.0, None))
    return frame


## 3. Temporal Backtests For Pure Baselines

Этот слой валиден для deterministic baseline. Для GRU/full stack ниже используем OOF, чтобы не симулировать невозможный future retrain.


In [ ]:
baseline_rows = []
period_frames = {}
for year, month in VALID_PERIODS:
    fr = predict_198_204(year, month, include_target=True)
    fr = fr[(fr[TARGET] > 0) & fr['pred198'].gt(0) & fr['pred204'].gt(0)].copy()
    period = label(year, month)
    period_frames[period] = fr
    for pred_col, candidate_id in [('pred198', 198), ('pred204', 204)]:
        baseline_rows.append({
            'period': period,
            'candidate_id': candidate_id,
            'variant': pred_col,
            'rows': len(fr),
            'mape': mape(fr[TARGET], fr[pred_col]),
            'wape': wape(fr[TARGET], fr[pred_col]),
            'log_mae_x100': log_mae(fr[TARGET], fr[pred_col]),
            'mean_pred': float(fr[pred_col].mean()),
        })

baseline_backtest = pd.DataFrame(baseline_rows)
baseline_backtest.to_csv(OUT_DIR / 'baseline_temporal_backtest.csv', index=False)
pivot = baseline_backtest.pivot(index='period', columns='candidate_id', values='mape').reset_index()
pivot['delta_204_vs_198'] = pivot[204] - pivot[198]
display(baseline_backtest)
display(pivot)


## 4. OOF Diagnostics For GRU And Second Residual

Здесь оцениваем только out-of-fold предсказания. Это главный anti-leakage слой для residual stack.


In [ ]:
def metric_pack(name, y, pred, ref=None):
    row = {
        'variant': name,
        'mape': mape(y, pred),
        'wape': wape(y, pred),
        'log_mae_x100': log_mae(y, pred),
        'mean_pred': float(np.mean(pred)),
    }
    if ref is not None:
        row['delta_mape_vs_ref'] = row['mape'] - mape(y, ref)
        row['mean_abs_change_vs_ref_pct'] = float(np.mean(np.abs(np.asarray(pred) / np.asarray(ref) - 1.0)) * 100)
    return row

oof_metric_rows = []
analysis_frame = None

gru_oof_path = RESULTS_DIR / 'temporal204_gru_residual' / 'oof.csv'
if gru_oof_path.exists():
    gru_oof = pd.read_csv(gru_oof_path)
    gru_oof['pred210_oof'] = gru_oof['new_base'].to_numpy(float) * np.exp(
        gru_oof['pred_log_gru'].to_numpy(float) - float(gru_oof['pred_log_gru'].median())
    )
    oof_metric_rows.append(metric_pack('204_base_oof', gru_oof[TARGET], gru_oof['new_base']))
    oof_metric_rows.append(metric_pack('210_gru_global_oof', gru_oof[TARGET], gru_oof['pred210_oof'], ref=gru_oof['new_base']))
    features_2024_03 = predict_198_204(2024, 3, include_target=True).drop(columns=[TARGET], errors='ignore')
    analysis_frame = gru_oof.merge(features_2024_03, on='new_id', how='left', suffixes=('', '_feat'))

second_oof_path = RESULTS_DIR / 'temporal204_second_residual' / 'oof.csv'
if second_oof_path.exists():
    second_oof = pd.read_csv(second_oof_path)
    second_center = second_oof['pred_second_log_cb'] - float(second_oof['pred_second_log_cb'].median())
    second_oof['pred217_tiny_on210_oof'] = second_oof['pred210'] * np.exp(np.clip(0.10 * second_center, -0.005, 0.005))
    second_oof['pred218_mid_on210_oof'] = second_oof['pred210'] * np.exp(np.clip(0.20 * second_center, -0.008, 0.008))
    second_oof['pred_best_oof_grid_on210'] = second_oof['pred210'] * np.exp(np.clip(0.60 * second_center, -0.020, 0.020))
    oof_metric_rows.append(metric_pack('217_like_second_tiny_on210_oof', second_oof[TARGET], second_oof['pred217_tiny_on210_oof'], ref=second_oof['pred210']))
    oof_metric_rows.append(metric_pack('218_like_second_mid_on210_oof', second_oof[TARGET], second_oof['pred218_mid_on210_oof'], ref=second_oof['pred210']))
    oof_metric_rows.append(metric_pack('best_grid_second_on210_oof', second_oof[TARGET], second_oof['pred_best_oof_grid_on210'], ref=second_oof['pred210']))
    if analysis_frame is not None:
        analysis_frame = analysis_frame.merge(
            second_oof[['new_id', 'pred_second_log_cb', 'pred217_tiny_on210_oof', 'pred218_mid_on210_oof', 'pred_best_oof_grid_on210']],
            on='new_id',
            how='left',
        )

oof_metrics = pd.DataFrame(oof_metric_rows).sort_values('mape').reset_index(drop=True)
oof_metrics.to_csv(OUT_DIR / 'oof_metric_summary.csv', index=False)
display(oof_metrics)


## 5. Segment Robustness

Смотрим не только среднюю MAPE, но и худшие срезы, где residual layers могут ломаться.


In [ ]:
def add_qcut_bin(frame, col, q=5):
    out = frame[col].replace([np.inf, -np.inf], np.nan)
    if out.notna().sum() < 100 or out.nunique(dropna=True) < 3:
        return pd.Series(['__missing__'] * len(frame), index=frame.index)
    try:
        return pd.qcut(out, q=q, duplicates='drop').astype(str).fillna('__missing__')
    except ValueError:
        return pd.Series(['__missing__'] * len(frame), index=frame.index)

def segment_metrics(frame, segment_col, variants, ref_variant='new_base', min_n=200):
    rows = []
    for seg, part in frame.groupby(segment_col, observed=True):
        if len(part) < min_n:
            continue
        ref_mape = mape(part[TARGET], part[ref_variant])
        for variant in variants:
            rows.append({
                'segment_col': segment_col,
                'segment': str(seg),
                'variant': variant,
                'n': len(part),
                'mape': mape(part[TARGET], part[variant]),
                'wape': wape(part[TARGET], part[variant]),
                'delta_mape_vs_ref': mape(part[TARGET], part[variant]) - ref_mape,
                'mean_abs_change_vs_ref_pct': float(np.mean(np.abs(part[variant].to_numpy(float) / part[ref_variant].to_numpy(float) - 1.0)) * 100),
            })
    return rows

segment_report = pd.DataFrame()
if analysis_frame is not None:
    seg = analysis_frame.copy()
    seg['volume_bin'] = add_qcut_bin(seg, 'lag1', q=5)
    seg['volatility_bin'] = add_qcut_bin(seg, 'volatility_rel3', q=5)
    seg['momentum_raw_bin'] = add_qcut_bin(seg, 'momentum_raw', q=5)
    seg['temporal_corr_abs_bin'] = add_qcut_bin(seg, 'temporal_corr_log', q=5)
    seg['gru_corr_abs_bin'] = add_qcut_bin((seg.assign(gru_corr_abs=(seg['pred_log_gru'] - seg['pred_log_gru'].median()).abs())), 'gru_corr_abs', q=5)

    variants = ['new_base', 'pred210_oof']
    for maybe in ['pred217_tiny_on210_oof', 'pred218_mid_on210_oof', 'pred_best_oof_grid_on210']:
        if maybe in seg.columns:
            variants.append(maybe)

    segment_cols = [c for c in STATIC_CAT_COLS if c in seg.columns]
    segment_cols += ['volume_bin', 'volatility_bin', 'momentum_raw_bin', 'temporal_corr_abs_bin', 'gru_corr_abs_bin']
    rows = []
    for col in segment_cols:
        rows.extend(segment_metrics(seg, col, variants, ref_variant='new_base', min_n=200))
    segment_report = pd.DataFrame(rows)
    segment_report.to_csv(OUT_DIR / 'segment_oof_metrics.csv', index=False)

display(segment_report.sort_values(['variant', 'delta_mape_vs_ref']).head(30) if not segment_report.empty else 'No OOF segment report')
display(segment_report.sort_values(['variant', 'delta_mape_vs_ref']).groupby('variant').tail(5) if not segment_report.empty else '')


## 6. Paired Bootstrap And Group Bootstrap

Проверяем не только point estimate, но и вероятность, что дельта стабильно положительная.


In [ ]:
def paired_bootstrap_delta(y, ref, pred, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    pred = np.asarray(pred, dtype=float)
    ref_ape = np.abs((y - ref) / y) * 100
    pred_ape = np.abs((y - pred) / y) * 100
    delta = pred_ape - ref_ape
    idx = rng.integers(0, len(delta), size=(n_boot, len(delta)))
    boot = delta[idx].mean(axis=1)
    return {
        'delta_mean': float(delta.mean()),
        'boot_p05': float(np.quantile(boot, 0.05)),
        'boot_p50': float(np.quantile(boot, 0.50)),
        'boot_p95': float(np.quantile(boot, 0.95)),
        'prob_improve': float((boot < 0).mean()),
    }

def group_bootstrap_delta(frame, group_col, ref_col, pred_col, n_boot=2000, seed=SEED):
    group_values = frame[group_col].astype(str).fillna('__nan__')
    groups = np.array(sorted(group_values.unique()))
    group_codes = {value: idx for idx, value in enumerate(groups)}
    codes = group_values.map(group_codes).to_numpy(int)
    n_groups = len(groups)
    y = frame[TARGET].to_numpy(float)
    ref_ape = np.abs((y - frame[ref_col].to_numpy(float)) / y) * 100
    pred_ape = np.abs((y - frame[pred_col].to_numpy(float)) / y) * 100
    sizes = np.bincount(codes, minlength=n_groups).astype(float)
    ref_sum = np.bincount(codes, weights=ref_ape, minlength=n_groups)
    pred_sum = np.bincount(codes, weights=pred_ape, minlength=n_groups)
    rng = np.random.default_rng(seed)
    sampled = rng.integers(0, n_groups, size=(n_boot, n_groups))
    deltas = (pred_sum[sampled].sum(axis=1) - ref_sum[sampled].sum(axis=1)) / sizes[sampled].sum(axis=1)
    return {
        'group_col': group_col,
        'variant': pred_col,
        'groups': n_groups,
        'boot_p05': float(np.quantile(deltas, 0.05)),
        'boot_p50': float(np.quantile(deltas, 0.50)),
        'boot_p95': float(np.quantile(deltas, 0.95)),
        'prob_improve': float((deltas < 0).mean()),
    }

boot_rows = []
group_boot_rows = []
if analysis_frame is not None:
    boot_variants = ['pred210_oof'] + [c for c in ['pred217_tiny_on210_oof', 'pred218_mid_on210_oof', 'pred_best_oof_grid_on210'] if c in analysis_frame.columns]
    for col in boot_variants:
        boot_rows.append({'variant': col, **paired_bootstrap_delta(analysis_frame[TARGET], analysis_frame['new_base'], analysis_frame[col])})
    for group_col in [c for c in STATIC_CAT_COLS if c in analysis_frame.columns]:
        for col in boot_variants:
            group_boot_rows.append(group_bootstrap_delta(analysis_frame, group_col, 'new_base', col, n_boot=1000))

bootstrap_report = pd.DataFrame(boot_rows)
group_bootstrap_report = pd.DataFrame(group_boot_rows)
bootstrap_report.to_csv(OUT_DIR / 'paired_bootstrap_oof.csv', index=False)
group_bootstrap_report.to_csv(OUT_DIR / 'group_bootstrap_oof.csv', index=False)
display(bootstrap_report)
display(group_bootstrap_report)


## 7. Test Submission Delta Diagnostics

Без actuals смотрим только magnitude/shape: насколько кандидат меняет anchor и в каких срезах. Это не validation score, а risk diagnostic.


In [ ]:
def find_submission(candidate_id: int) -> Path | None:
    if not public_focus.empty:
        rows = public_focus[public_focus['candidate_id_num'] == candidate_id]
        if len(rows) and isinstance(rows.iloc[0].get('file'), str):
            p = SUBMISSIONS_DIR / rows.iloc[0]['file']
            if p.exists():
                return p
    matches = sorted(SUBMISSIONS_DIR.glob(f'stage2_candidate_{candidate_id:03d}_*.csv'))
    return matches[0] if matches else None

submission_preds = {}
for cid in CANDIDATE_IDS:
    path = find_submission(cid)
    if path is None:
        continue
    sub = pd.read_csv(path)
    submission_preds[cid] = sub.rename(columns={'rto': f'pred{cid}'})[['new_id', f'pred{cid}']]

test_frame = predict_198_204(2025, 3, include_target=False)
test_joined = test_frame[['new_id', 'lag1', 'momentum_raw', 'volatility_rel3', 'temporal_corr_log'] + [c for c in STATIC_CAT_COLS if c in test_frame.columns]].copy()
for cid, sub in submission_preds.items():
    test_joined = test_joined.merge(sub, on='new_id', how='left')

delta_rows = []
for cid in submission_preds:
    for ref in [204, 210, 215, 220]:
        if cid == ref or f'pred{ref}' not in test_joined or f'pred{cid}' not in test_joined:
            continue
        ratio = test_joined[f'pred{cid}'].to_numpy(float) / test_joined[f'pred{ref}'].to_numpy(float) - 1.0
        delta_rows.append({
            'candidate_id': cid,
            'ref_candidate_id': ref,
            'mean_delta_pct': float(np.mean(ratio) * 100),
            'mean_abs_delta_pct': float(np.mean(np.abs(ratio)) * 100),
            'p95_abs_delta_pct': float(np.quantile(np.abs(ratio), 0.95) * 100),
            'max_abs_delta_pct': float(np.max(np.abs(ratio)) * 100),
            'corr': float(np.corrcoef(test_joined[f'pred{cid}'], test_joined[f'pred{ref}'])[0, 1]),
        })
delta_report = pd.DataFrame(delta_rows)
delta_report.to_csv(OUT_DIR / 'test_submission_delta_diagnostics.csv', index=False)
display(delta_report.sort_values(['ref_candidate_id', 'mean_abs_delta_pct']))


## 8. Shift / Adversarial Validation

Проверяем, насколько March-2025 target rows похожи на historical validation rows по доступным признакам. Чем легче отличить target, тем осторожнее доверяем historical proxy.


In [ ]:
shift_rows = []
for year, month in VALID_PERIODS:
    fr = predict_198_204(year, month, include_target=True)
    fr['domain'] = 0
    fr['period'] = label(year, month)
    shift_rows.append(fr)
target_shift = predict_198_204(2025, 3, include_target=False)
target_shift['domain'] = 1
target_shift['period'] = '2025-03_target'
shift_rows.append(target_shift)
shift_frame = pd.concat(shift_rows, ignore_index=True)

shift_features_num = [
    'lag1', 'lag2', 'lag3', 'lag12', 'lag13', 'rolling3_mean', 'momentum_raw',
    'feb_to_jan', 'jan_to_dec', 'lag1_to_lag12', 'lag1_to_lag13',
    'volatility_rel3', 'store_mar_feb_ratio_prev', 'temporal_corr_log',
]
shift_features_cat = [c for c in STATIC_CAT_COLS if c in shift_frame.columns]

try:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    X = shift_frame[shift_features_num + shift_features_cat].copy()
    y = shift_frame['domain'].astype(int).to_numpy()
    pre = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), shift_features_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=50), shift_features_cat),
    ])
    clf = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=500, C=0.5, n_jobs=None))])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    pred_domain = cross_val_predict(clf, X, y, cv=cv, method='predict_proba')[:, 1]
    auc = roc_auc_score(y, pred_domain)
    shift_frame['domain_pred_oof'] = pred_domain
    shift_summary = shift_frame.groupby('period', as_index=False).agg(
        rows=('new_id', 'size'),
        mean_domain_pred=('domain_pred_oof', 'mean'),
        p95_domain_pred=('domain_pred_oof', lambda s: float(np.quantile(s, 0.95))),
    )
    print('Adversarial validation AUC target-vs-history:', round(float(auc), 5))
    display(shift_summary)
    shift_summary.to_csv(OUT_DIR / 'shift_adversarial_validation_summary.csv', index=False)
except Exception as exc:
    print('Shift classifier skipped:', repr(exc))
    shift_summary = pd.DataFrame()


## 9. Decision Scorecard

Этот scorecard не заменяет public score. Он нужен, чтобы перед сабмитом понимать: идея новая или микротюнинг, где риск, насколько велика коррекция и есть ли OOF/segment подтверждение.


In [ ]:
scorecard = registry.copy()
if 'public_mape' in scorecard.columns:
    scorecard['public_mape_num'] = pd.to_numeric(scorecard['public_mape'], errors='coerce')

# OOF layer mapping is intentionally conservative: 220/221 exact OOF on 215 is unavailable here.
oof_lookup = {}
if not oof_metrics.empty:
    for _, row in oof_metrics.iterrows():
        oof_lookup[row['variant']] = row.to_dict()
scorecard['oof_proxy_note'] = ''
scorecard.loc[scorecard['candidate_id'] == 204, 'oof_proxy_note'] = 'baseline temporal backtest + base OOF anchor'
scorecard.loc[scorecard['candidate_id'] == 210, 'oof_proxy_note'] = 'direct GRU OOF available'
scorecard.loc[scorecard['candidate_id'].isin([220, 221]), 'oof_proxy_note'] = 'second residual OOF only on 210; public candidate applies to 215'

if not delta_report.empty:
    vs215 = delta_report[delta_report['ref_candidate_id'] == 215][['candidate_id', 'mean_abs_delta_pct', 'p95_abs_delta_pct', 'max_abs_delta_pct']]
    vs215 = vs215.rename(columns={
        'mean_abs_delta_pct': 'mean_abs_change_vs_215_pct',
        'p95_abs_delta_pct': 'p95_abs_change_vs_215_pct',
        'max_abs_delta_pct': 'max_abs_change_vs_215_pct',
    })
    scorecard = scorecard.merge(vs215, on='candidate_id', how='left')

scorecard['submit_policy'] = np.select(
    [
        scorecard['candidate_id'].eq(220),
        scorecard['candidate_id'].eq(221),
        scorecard['candidate_id'].eq(215),
        scorecard['candidate_id'].eq(210),
        scorecard['candidate_id'].eq(204),
    ],
    [
        'current best; use as public anchor, verify private-safe via scorecard',
        'do not prefer over 220 unless new validation signal appears',
        'strong one-layer fallback; compare private-safe vs 220',
        'clean GRU layer; useful if transfer/second residual look risky',
        'pure deterministic fallback and baseline for new modeling',
    ],
    default='diagnostic/reference',
)

scorecard.to_csv(OUT_DIR / 'decision_scorecard.csv', index=False)
display(scorecard.sort_values(['public_mape_num', 'candidate_id'], na_position='last'))


## 10. Submission Discipline For Next Score Push

Чтобы добить `90.8+`, public submissions должны идти только после lab-gate:

1. Новая независимая идея, а не neighbor tuning вокруг `220/221`.
2. Есть OOF или temporal evidence.
3. Нет крупного worst-slice ухудшения.
4. Коррекция относительно `215/220` мала или хорошо объяснена сегментно.
5. Adversarial validation показывает, где proxy может быть ненадежен, и кандидат не опирается только на эти срезы.

Кандидаты для будущего усиления лучше искать в новых независимых слоях:

- multi-seed / small ensemble fresh GRU on `204`;
- OOF-safe blending of `210`, old-transfer `118`, and second residual with learned or rule-based shrinkage;
- deterministic feature-based ensemble around temporal/Q1, Q1 residual, local/group shrinkage;
- segment-specific dampening where `220` improves public but looks fragile.
